# The clause tree — what every extraction is built on

The clauses of the graph come from the tree `build_clause_tree` reads off the numbering, stored
in each paragraph file. This notebook checks that tree for every contract, with no API call:

1. the stored tree against the one the current builder makes — a difference means the file
   predates a fix and needs regenerating;
2. what the tree misses: paragraphs after the last numbered clause (exhibits, signatures) fall
   into that clause;
3. the blocks the extraction would make, and the context each one reads.

The two fixes of 2026-10-05: a `)`-numbered paragraph is a list item, not a heading (one of
them, "2) inspection of…", cut Bellicum's tree at 9.4 of 20 articles), and a heading followed
by a comma is a reference split by a page break ("13.1 , Bellicum shall…").

In [1]:
import sys, os, json
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'server').exists():
    ROOT = ROOT.parent
SERVER = ROOT / 'server'
if str(SERVER) not in sys.path:
    sys.path.insert(0, str(SERVER))

from services.documents.processing import build_clause_tree
from services.graph.knowledge.extraction import Outline, plan_blocks

def flatten(tree):
    out = []
    def walk(nodes):
        for node in nodes:
            out.append(node)
            walk(node.get('children') or [])
    walk(tree)
    return out

files = sorted((ROOT / 'infra/json/paragraphs').glob('*.json'))

## 1. Stored tree against rebuilt tree

In [2]:
for path in files:
    data = json.loads(path.read_text())
    paragraphs = data['paragraphs']
    index = {p['id']: i for i, p in enumerate(paragraphs)}
    stored, rebuilt = flatten(data['tree']), flatten(build_clause_tree(paragraphs))
    def describe(nodes):
        refs = [n['ref'] for n in nodes]
        tops = [r for r in refs if '.' not in r]
        twins = sorted({r for r in refs if refs.count(r) > 1})
        last = max((index[n['paragraphId']] for n in nodes), default=-1)
        return f"{len(nodes):3} nodes, articles {tops[0] if tops else '-'}..{tops[-1] if tops else '-'}, last at paragraph {last}/{len(paragraphs)}, repeated {twins or 'none'}"
    same = [(n['ref'], n['paragraphId']) for n in stored] == [(n['ref'], n['paragraphId']) for n in rebuilt]
    print(f"{path.stem[5:45]}\n   stored : {describe(stored)}\n   rebuilt: {describe(rebuilt)}{'' if same else '   <- needs regenerating'}")

BELLICUMPHARMACEUTICALS_INC_05_07_2019-E
   stored : 120 nodes, articles 1..20, last at paragraph 375/403, repeated none
   rebuilt: 120 nodes, articles 1..20, last at paragraph 375/403, repeated none
EdietsComInc_20001030_10QSB_EX-10_4_2606
   stored :  68 nodes, articles 1..15, last at paragraph 125/331, repeated none
   rebuilt:  68 nodes, articles 1..15, last at paragraph 125/331, repeated none
HealthcentralCom_19991108_S-1A_EX-10_27_
   stored :  31 nodes, articles 1..28, last at paragraph 166/208, repeated none
   rebuilt:  31 nodes, articles 1..28, last at paragraph 166/208, repeated none
RitterPharmaceuticalsInc_20200313_S-4A_E
   stored : 134 nodes, articles 1..15, last at paragraph 197/522, repeated ['3.3']
   rebuilt: 134 nodes, articles 1..15, last at paragraph 197/522, repeated ['3.3']
SteelVaultCorp_20081224_10-K_EX-10_16_30
   stored :  26 nodes, articles 1..12, last at paragraph 39/52, repeated none
   rebuilt:  26 nodes, articles 1..12, last at paragraph 39/52, repeate

## 2. What falls after the last clause

Exhibits and signature blocks have no number of their own, so they belong to the last numbered
clause. When they carry obligations — Ediets' exhibits guarantee ad impressions — those
statements would be filed under a clause like *Entire Agreement*.

In [3]:
for path in files:
    data = json.loads(path.read_text())
    paragraphs = data['paragraphs']
    index = {p['id']: i for i, p in enumerate(paragraphs)}
    nodes = flatten(build_clause_tree(paragraphs))
    last = max((index[n['paragraphId']] for n in nodes), default=-1)
    tail = [p for p in paragraphs[last + 1:] if p['text'].strip()]
    print(f"{path.stem[5:45]}: {len(tail)} paragraphs after the last clause ({sum(len(p['text']) for p in tail):,} characters)")
    for p in tail[:: max(1, len(tail) // 4)][:4]:
        print(f"    {p['text'][:90]}")

BELLICUMPHARMACEUTICALS_INC_05_07_2019-E: 27 paragraphs after the last clause (2,355 characters)
    [Remainder of this page intentionally left blank. Signature page follows.]
    By: /s/ Rick Fair
    EXHIBIT D [...***...] Sublicense Royalties and/or License Fees EXHIBIT E Product Prices
    Within the scope of the [...***...] License Agreement, Miltenyi has got the right to grant
EdietsComInc_20001030_10QSB_EX-10_4_2606: 205 paragraphs after the last clause (14,693 characters)
    Intent dated April 3, 2000, and any other agreements and communications (both
    www.prevention.com/weight/success.html
    o Resolutions Promotion (Year 2001)
    Other (Promotions, production, etc) $7,083
HealthcentralCom_19991108_S-1A_EX-10_27_: 41 paragraphs after the last clause (3,140 characters)
    The section and article headings contained in this Agreement are included for convenience 
    Schedule "2"
    Issues related to access to services:
    Different health events e.g. "Heart month"
Ritter

## 3. The blocks an extraction would make

In [4]:
STEM = 'root_SteelVaultCorp_20081224_10-K_EX-10_16_3074935_EX-10_16_Affiliate_Agreement'
data = json.loads((ROOT / 'infra/json/paragraphs' / f'{STEM}.json').read_text())
outline = Outline(data['tree'], data['paragraphs'])
texts = {i: p['text'].strip() for i, p in enumerate(data['paragraphs']) if p['text'].strip()}
for n, block in enumerate(plan_blocks(outline, texts), start=1):
    size = sum(len(texts[i]) + 20 for i in block.extract)
    clauses = []
    for i in block.extract:
        section = outline.section_at(i)
        ref = section.ref if section else 'preamble'
        if ref not in clauses:
            clauses.append(ref)
    print(f'block {n}: paragraphs {block.extract[0]}-{block.extract[-1]}, {size:,} characters, clauses {clauses} | context {block.context}')

block 1: paragraphs 0-9, 4,659 characters, clauses ['preamble', '1'] | context [10]
block 2: paragraphs 10-23, 4,753 characters, clauses ['2', '3', '4', '5', '5.1', '5.2', '5.3', '5.4', '5.5', '5.6', '6'] | context [9, 24]
block 3: paragraphs 24-37, 4,185 characters, clauses ['7', '7.1', '7.2', '7.3', '7.4', '8', '8.1', '8.2', '8.3', '8.4', '9', '10'] | context [23, 38]
block 4: paragraphs 38-51, 4,810 characters, clauses ['11', '12'] | context [37]
